# NóiLái — Kaggle 2×T4 evaluation runs

Runs one line of `configs/run_plan.yaml` (default `E1_main`) over a list of models from
`configs/models.yaml` with `scripts/run_eval.py --resume`, on Kaggle's free 2×T4 session
(12 h max, ~30 GPU-h/week). Every cell is non-interactive: set the **parameters** cell,
then *Run all*. Attach before starting:

* accelerator **GPU T4 ×2**; internet on;
* Kaggle Secrets: `GITHUB_TOKEN` (repository read, only if no bundle), `HF_TOKEN`
  (gated Gemma/Llama/Vistral), `KAGGLE_USERNAME` + `KAGGLE_KEY` (dataset push);
* a private dataset holding `noilai-main.bundle` (`git bundle create noilai-main.bundle main`)
  and one holding the frozen `data/release/` (never public: the test split is gated).

Cells: (a) environment · (b) clone · (c) pinned install · (d) resources + item hash ·
(e) smoke test · (f) **RUN** · (g) outputs → dataset · (h) GPU-hours log.
The flag mapping to `run_eval.py` lives in `scripts/kaggle_run_plan.py`; adjust flags there
or through `EXTRA_ARGS`. Kaggle's own environment (nvidia driver, CUDA) is recorded by (c).

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/OWNER/REPO.git"     # [UNCERTAIN: verify] the repository that holds noilai/ (or the split-out noilai repo)
REPO_REF = "main"                                   # pin a commit hash for a paper run; recorded in every manifest
REPO_SUBDIR = "noilai"                              # "" once noilai is a repository of its own
BUNDLE_PATH = "/kaggle/input/noilai-bundle/noilai-main.bundle"   # git bundle in a private dataset; preferred over a token
CLONE_DIR = "/kaggle/working/repo"
WORK_DIR = "/kaggle/working"                        # ~20 GB, saved with the notebook
HF_HOME = "/kaggle/tmp/hf"                          # model cache outside the working dir [UNCERTAIN: verify the quota of /kaggle/tmp]
ITEMS_DATASET_DIR = "/kaggle/input/noilai-release"  # private dataset with the frozen data/release/<version>/ tree; None if the clone has it
VERIFY_ITEM_KEYS = ["noilai_test", "noilai_dev"]    # keys of configs/run_plan.yaml item_files checked in (d)

VLLM_VERSION = "0.11.0"          # [UNCERTAIN: verify] pin to the version the smoke tests pass with; recorded by (c)
TRANSFORMERS_VERSION = None      # None = whatever the vllm wheel requires (pip freeze is recorded either way)

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

RUN_ID = "E1_main"               # a run id from configs/run_plan.yaml (kaggle_run_plan.py --list)
MODELS = ["gemma-3-1b-it"]       # subset of the run's models; [] = every model of the run
SMOKE_MODEL = "gemma-3-1b-it"    # (e) 20 items on this model before the run
EXTRA_ARGS = ""                  # extra run_eval.py flags, one shell-quoted string, e.g. "--seed 7"
DRY_RUN = False                  # True prints the commands and runs nothing

PLATFORM, GPU_TYPE, N_GPUS = "kaggle", "t4", 2
REQUIRE_GPU = True
RUN_LABEL = RUN_ID
PUSH_DATASET = True
KAGGLE_DATASET_SLUG = "noilai-runs"
CREATE_DATASET = False           # True the first time the dataset is published

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# (a) environment check: interpreter, GPU(s), CUDA, vLLM's compute-capability floor (7.5), disk
import json, shutil, subprocess, sys
print(sys.version.split()[0], "on", sys.platform)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found: is a GPU accelerator attached?")
ENV_CHECK = {"n_gpus": 0, "devices": [], "vllm_ok": False}
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        cap = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 2**30
        ENV_CHECK["devices"].append({"name": torch.cuda.get_device_name(i), "capability": f"{cap[0]}.{cap[1]}", "memory_gb": round(mem, 1)})
        print(f"  gpu{i}: {torch.cuda.get_device_name(i)}  capability {cap[0]}.{cap[1]}  {mem:.1f} GB")
    ENV_CHECK["n_gpus"] = torch.cuda.device_count()
    ENV_CHECK["vllm_ok"] = ENV_CHECK["n_gpus"] > 0 and all(
        tuple(int(x) for x in d["capability"].split(".")) >= (7, 5) for d in ENV_CHECK["devices"])
except ImportError:
    print("torch is not importable yet (installed in the install cell)")
if REQUIRE_GPU and ENV_CHECK["n_gpus"] == 0:
    raise SystemExit("no CUDA device: choose the GPU accelerator (Kaggle: 'GPU T4 x2') and restart")
if ENV_CHECK["n_gpus"] and not ENV_CHECK["vllm_ok"]:
    print("WARNING: a device is below compute capability 7.5 (P100?) -> vLLM cannot run; use backend hf / llama.cpp")
du = shutil.disk_usage(WORK_DIR)
print(f"disk at {WORK_DIR}: {du.free / 2**30:.1f} GB free of {du.total / 2**30:.1f} GB")
print(json.dumps(ENV_CHECK))

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
        env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
for cand in (REPO_REF, f"origin/{REPO_REF}"):
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF}")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (c) pinned install: the inference engine first so that its own requirements win, then the project
# with its eval extra; then record pip freeze + versions next to the outputs (checklist C4).
import subprocess, sys
from pathlib import Path
pins = [f"vllm=={VLLM_VERSION}"]
if TRANSFORMERS_VERSION:
    pins.append(f"transformers=={TRANSFORMERS_VERSION}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=True)
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
_export(HF_TOKEN_SECRET)          # gated checkpoints (Gemma, Llama, Vistral); exported as HF_TOKEN when the secret has that name
if HF_TOKEN_SECRET != "HF_TOKEN" and os.environ.get(HF_TOKEN_SECRET):
    os.environ["HF_TOKEN"] = os.environ[HF_TOKEN_SECRET]
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s): hash against
# configs/run_plan.yaml, canary on every row, row count. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    shutil.copytree(src, PROJECT / "data" / "release", dirs_exist_ok=True)
    print("release copied from", src)
for key in VERIFY_ITEM_KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

### RUN CELL (e) — smoke test
One small model, 20 items (`smoke_20` in `run_plan.yaml`): load, generate, log-probs. Stops the
notebook on failure so that no GPU hour is spent on a broken environment.

In [ ]:
# RUN CELL (e): smoke test. Flags come from run_plan.yaml (smoke_20) via scripts/kaggle_run_plan.py.
import shlex
import kaggle_run_plan as KRP
SMOKE_RESULTS = KRP.execute("smoke_20", models=[SMOKE_MODEL], platform=PLATFORM, dry_run=DRY_RUN,
                            continue_on_error=False, extra=shlex.split(EXTRA_ARGS))
for r in SMOKE_RESULTS:
    print(r["model"], "->", r["status"])
if any(str(r["status"]).startswith("failed") for r in SMOKE_RESULTS):
    raise SystemExit("smoke test failed: fix the environment or the model entry before the run cell")

### RUN CELL (f) — the run
Loops over `MODELS` (or every model of `RUN_ID`) calling
`scripts/run_eval.py --model-config <name> --items ... --tasks ... --arms ... --resume --run-id <RUN_ID>__<name> --out-root data/runs`.
`--resume` makes a re-run after a session timeout continue where it stopped. Each model's wall
time is appended to `data/compute_log.csv` with the device type from `models.yaml`.

In [ ]:
# RUN CELL (f): the run matrix line RUN_ID over MODELS, with --resume. Adjust flags in
# scripts/kaggle_run_plan.py (build_command) or through EXTRA_ARGS; the CLI is owned by noilai/eval.
import json, shlex
import kaggle_run_plan as KRP
RESULTS = KRP.execute(RUN_ID, models=MODELS or None, platform=PLATFORM, dry_run=DRY_RUN,
                      continue_on_error=True, extra=shlex.split(EXTRA_ARGS))
print(json.dumps([{"model": r["model"], "status": r["status"], "hours": r.get("hours"),
                   "new_outputs": r.get("new_outputs")} for r in RESULTS], indent=1))

In [ ]:
# (g) keep the outputs: copy data/runs (+ compute log) into /kaggle/working (saved with the notebook) and
# add a version to a private Kaggle dataset through scripts/kaggle_dataset.py (credentials from Secrets).
import os, shutil, subprocess, sys
from pathlib import Path
out = Path(WORK_DIR) / "noilai_runs_out"
shutil.copytree(PROJECT / "data" / "runs", out / "runs", dirs_exist_ok=True)
if (PROJECT / "data" / "compute_log.csv").exists():
    shutil.copy2(PROJECT / "data" / "compute_log.csv", out / "compute_log.csv")
print("outputs copied to", out)
if PUSH_DATASET:
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        _export(name)
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        cmd = [sys.executable, "scripts/kaggle_dataset.py", "--src", str(PROJECT / "data" / "runs"),
               "--src", str(PROJECT / "data" / "compute_log.csv"), "--dest", str(Path(WORK_DIR) / "noilai_runs_dataset"),
               "--slug", KAGGLE_DATASET_SLUG, "--message", f"{RUN_LABEL} @ {HEAD[:8]}"]
        if CREATE_DATASET:
            cmd.append("--create")
        subprocess.run(cmd, check=False)
    else:
        print("Kaggle credentials not found: dataset push skipped (outputs are still under", out, ")")

In [ ]:
# (h) GPU-hours log (checklist C1). kaggle_run_plan wrote one row per executed model run; this adds the
# session's remaining wall time (startup, install, model downloads) as "overhead" and prints the totals.
import shutil
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
run_hours = sum((r.get("hours") or 0.0) for r in (globals().get("RESULTS", []) + globals().get("SMOKE_RESULTS", [])))
overhead = max(0.0, CL.hours_since(SESSION_T0) - run_hours)
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=overhead, run_id=f"session-{RUN_LABEL}", purpose="overhead"))
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
out = Path(WORK_DIR) / "noilai_runs_out"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOG, out / "compute_log.csv")
print("log copied to", out / "compute_log.csv", "-> merge into the repository's data/compute_log.csv")